# Phenoconversion Timing Estimation — UKB (Horizon Shifting)
## Applying Miami-Trained Tobit Models with Individual-Level Time Shifting

**Author:** Ximing Ran



## Overview

This notebook applies the Miami-trained Tobit regression models to UK Biobank (UKB) **Convert** delta protein data using **individual-level time shifting** across six pre-diagnosis horizons.

**Analysis group:** UKB Convert individuals with known `YrSinceDi` (years since diagnosis at blood collection, negative = pre-diagnosis).

**Horizons:** `{−5, −4, −3, −2, −1, 0}` years (relative to the individual's own `YrSinceDi`)



## Time-Shifting Logic

For each horizon `h ∈ {−5, −4, −3, −2, −1, 0}`, each individual's shifted time is:

$$Y_{\text{shifted}} = \text{YrSinceDi} - h$$

Individuals where $Y_{\text{shifted}} \geq 0$ are **dropped** (the shift would place them at or after diagnosis).

**Concrete example:**

| `YrSinceDi` | Horizon `h` | `y_true = YrSinceDi − h` | Kept? |
|---|---|---|---|
| −7 | −2 | −7 − (−2) = **−5** | ✓ |
| −1 | −2 | −1 − (−2) = **+1** | ✗ dropped |
| −3 | −5 | −3 − (−5) = **+2** | ✗ dropped |
| −6 | −5 | −6 − (−5) = **−1** | ✓ |

This means each horizon asks: *"For individuals who were sampled at least |h| years before diagnosis, what does the model predict if we shift their reference point by h years?"* The resulting `y_true` is patient-specific (not a fixed constant), so individuals at different original `YrSinceDi` values contribute different shifted ground truths within the same horizon.



## Key Adaptations from Miami Training

| Aspect | Miami (training) | UKB (application) |
|--------|-----------------|-------------------|
| Groups | Convert (pre-onset visits, `YrSinceOs < 0`) | Convert with known `YrSinceDi` |
| Time variable | `YrSinceOs` | `YrSinceDi − h` (horizon-shifted) |
| Repeated visits | Yes — multiple per patient | No — one baseline visit |
| `_Init` columns | Earliest visit protein level | Same as current (cross-sectional) |
| Genotype coding | SOD1 A4V / SOD1 nonA4V / C9orf72 | Mapped directly from UKB `GenoGroup` |
| Missing proteins | N/A | Set to 0 (imputed) |


## 1. Load Libraries

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggrepel)
library(patchwork)
library(knitr)
library(kableExtra)
library(AER)
library(here)

set.seed(2024)
theme_set(theme_bw() + theme(legend.position = "bottom"))

# Fixed horizons (years before diagnosis, negative = pre-diagnosis)
HORIZONS <- c(-5, -4, -3, -2, -1, 0)


## 2. Helper Functions

Six diagnostic plot functions (identical layout to the Miami Rmd) plus `compute_metrics`:

- **`Bland_Altman_plot`** — agreement between observed and predicted, with ±1.96 SD limits of agreement
- **`residual_hist`** — histogram of prediction errors
- **`predict_vs_observed_plot_tobit`** — scatter with ±1-year tolerance band and annotation of MAE, RMSE, Correlation, R²
- **`predict_vs_observed_boxplot`** — observed values grouped by integer-binned predictions
- **`residual_vs_predict`** — residuals vs predicted values scatter
- **`residual_vs_predict_bin`** — residuals vs predicted values (binned boxplot)
- **`compute_metrics`** — returns MAE, RMSE, Pearson correlation, and simple R²; returns `NA` for all metrics if fewer than 3 complete pairs exist


In [ ]:
residual_hist <- function(y_true, y_predict, subtittle = NULL) {
  residuals <- y_true - y_predict
  ggplot(data.frame(residuals = residuals), aes(x = residuals)) +
    geom_histogram(binwidth = 0.5, fill = "lightblue", color = "black") +
    labs(title = "Histogram of Residuals", subtitle = subtittle,
         x = "Residuals (Observed - Predicted)", y = "Frequency") +
    theme_classic() +
    theme(plot.title    = element_text(hjust = 0.5, face = "bold", size = 14),
          plot.subtitle = element_text(hjust = 0.5, size = 12))
}

Bland_Altman_plot <- function(y_true, y_predict, subtittle = NULL) {
  mean_values <- (y_true + y_predict) / 2
  diff_values <- y_true - y_predict
  mean_diff   <- mean(diff_values, na.rm = TRUE)
  sd_diff     <- sd(diff_values,   na.rm = TRUE)
  loa_upper   <- mean_diff + 1.96 * sd_diff
  loa_lower   <- mean_diff - 1.96 * sd_diff
  plot_df     <- data.frame(mean_values = mean_values, diff_values = diff_values)
  ggplot(plot_df, aes(x = mean_values, y = diff_values)) +
    geom_point(alpha = 0.8) +
    geom_hline(yintercept = mean_diff,  color = "red",  linetype = "dashed", linewidth = 1) +
    geom_hline(yintercept = loa_upper,  color = "blue", linetype = "dashed", linewidth = 1) +
    geom_hline(yintercept = loa_lower,  color = "blue", linetype = "dashed", linewidth = 1) +
    labs(title = "Bland-Altman Plot", subtitle = subtittle,
         x = "Mean of Observed and Predicted", y = "Difference (Observed - Predicted)") +
    theme_classic() +
    theme(plot.title    = element_text(hjust = 0.5, face = "bold", size = 14),
          plot.subtitle = element_text(hjust = 0.5, size = 12)) +
    annotate("text",
             x     = max(plot_df$mean_values, na.rm = TRUE),
             y     = c(mean_diff, loa_upper, loa_lower) + c(0.15, 0.15, -0.15),
             label = c("Mean Diff", "+1.96 SD", "-1.96 SD"),
             color = c("red", "blue", "blue"), hjust = 1, size = 3.5)
}

predict_vs_observed_plot_tobit <- function(y_true, y_predict, subtitle = NULL,
                                           vjust = 0.75,
                                           MAE, RMSE, cor_y, R_squared) {
  df          <- data.frame(y_true = y_true, y_predict = y_predict)
  df$category <- ifelse(df$y_predict < df$y_true,
                        "Earlier than Predicted", "Later than Predicted")
  df$residuals <- df$y_true - df$y_predict
  prop_1       <- sum(abs(df$residuals) < 1, na.rm = TRUE) / nrow(df)
  range_limits <- range(c(y_true, y_predict), na.rm = TRUE)
  band_x       <- seq(range_limits[1], range_limits[2], length.out = 100)
  band_df      <- data.frame(x = c(band_x, rev(band_x)),
                             y = c(band_x - 1, rev(band_x + 1)))
  ggplot(df, aes(x = y_predict, y = y_true, color = category)) +
    geom_polygon(data = band_df, aes(x = x, y = y),
                 fill = "grey80", alpha = 0.5, inherit.aes = FALSE) +
    geom_point(alpha = 0.7, size = 2) +
    scale_color_manual(values = c("Earlier than Predicted" = "#D73027",
                                  "Later than Predicted"   = "#4575B4")) +
    geom_line(data = data.frame(x = band_x, y = band_x),
              aes(x = x, y = y), color = "black", linetype = "dashed",
              linewidth = 1, inherit.aes = FALSE) +
    labs(title = "Predicted vs. Observed", subtitle = subtitle,
         x = "Predicted YrSinceOs", y = "Observed (Horizon)", color = NULL) +
    theme_classic() +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
          plot.subtitle = element_text(hjust = 0.5, size = 12),
          legend.position = "top") +
    annotate("text", x = range_limits[1], y = range_limits[2],
             hjust = 0, vjust = vjust,
             label = paste0("MAE = ",  round(MAE, 3),
                            "\nRMSE = ", round(RMSE, 3),
                            "\n|Res|<1 = ", round(prop_1 * 100, 1), "%",
                            "\nCor = ",  round(cor_y, 3),
                            "\nR² = ",   round(R_squared, 3)),
             size = 4, fontface = "bold", color = "black")
}

predict_vs_observed_boxplot <- function(y_true, y_predict, subtitle = NULL) {
  df          <- data.frame(y_true = y_true, y_predict = y_predict)
  df$bin      <- round(df$y_predict)
  df$category <- ifelse(df$y_predict < df$y_true,
                        "Earlier than Predicted", "Later than Predicted")
  y_x_data    <- data.frame(bin = unique(df$bin), true = unique(df$bin))
  ggplot(df, aes(x = factor(bin), y = y_true)) +
    geom_boxplot(outlier.shape = NA, fill = "grey", alpha = 0.6) +
    geom_jitter(aes(color = category), width = 0.2, alpha = 0.7, size = 2) +
    geom_line(data = y_x_data, aes(x = factor(bin), y = true, group = 1),
              color = "black", linetype = "dashed", linewidth = 1) +
    scale_color_manual(values = c("Earlier than Predicted" = "#D73027",
                                  "Later than Predicted"   = "#4575B4")) +
    labs(title = "Predicted vs. Observed (Binned)", subtitle = subtitle,
         x = "Predicted Value (Binned)", y = "Observed (Horizon)", color = NULL) +
    theme_classic() +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
          plot.subtitle = element_text(hjust = 0.5, size = 12),
          legend.position = "top")
}

residual_vs_predict <- function(y_true, y_predict, subtitle = NULL) {
  residuals <- y_true - y_predict
  int_range <- seq(floor(min(y_predict, na.rm = TRUE)),
                   ceiling(max(y_predict, na.rm = TRUE)), by = 1)
  cat_col   <- ifelse(residuals > 0, "Earlier than Predicted", "Later than Predicted")
  ggplot(data.frame(y_predict = y_predict, residuals = residuals, cat = cat_col),
         aes(x = y_predict, y = residuals, color = cat)) +
    geom_vline(xintercept = int_range, color = "grey70", linetype = "dotted") +
    geom_point(alpha = 0.7, size = 2) +
    geom_hline(yintercept = 0, color = "black", linetype = "dashed", linewidth = 1) +
    scale_color_manual(values = c("Earlier than Predicted" = "#D73027",
                                  "Later than Predicted"   = "#4575B4")) +
    scale_x_continuous(breaks = int_range) +
    labs(title = "Residuals vs. Predicted", subtitle = subtitle,
         x = "Predicted Values", y = "Residuals (Observed - Predicted)", color = NULL) +
    theme_classic() +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
          plot.subtitle = element_text(hjust = 0.5, size = 12),
          legend.position = "top")
}

residual_vs_predict_bin <- function(y_true, y_predict, subtitle = NULL) {
  residuals <- y_true - y_predict
  df        <- data.frame(y_predict = y_predict, residuals = residuals)
  df$bin    <- round(df$y_predict)
  df$cat    <- ifelse(df$residuals > 0, "Earlier than Predicted", "Later than Predicted")
  ggplot(df, aes(x = as.factor(bin), y = residuals)) +
    geom_boxplot(outlier.shape = NA, fill = "gray80", alpha = 0.6) +
    geom_jitter(aes(color = cat), width = 0.2, alpha = 0.7, size = 2) +
    geom_hline(yintercept = 0, color = "black", linetype = "dashed", linewidth = 1) +
    scale_color_manual(values = c("Earlier than Predicted" = "#D73027",
                                  "Later than Predicted"   = "#4575B4")) +
    labs(title = "Residuals vs. Predicted (Binned)", subtitle = subtitle,
         x = "Rounded Predicted Values", y = "Residuals", color = NULL) +
    theme_classic() +
    theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
          plot.subtitle = element_text(hjust = 0.5, size = 12),
          legend.position = "top")
}

compute_metrics <- function(y_true, y_predict) {
  mask <- !is.na(y_true) & !is.na(y_predict)
  if (sum(mask) < 3) {
    return(list(MAE = NA_real_, RMSE = NA_real_,
                cor_y = NA_real_, R_squared = NA_real_))
  }
  y_t   <- y_true[mask]
  y_p   <- y_predict[mask]
  resid <- y_t - y_p
  y_mean <- mean(y_t, na.rm = TRUE)
  list(
    MAE       = mean(abs(resid)),
    RMSE      = sqrt(mean(resid^2)),
    cor_y     = cor(y_t, y_p),
    R_squared = 1 - sum(resid^2) / sum((y_t - y_mean)^2)
  )
}

## 3. Load Miami-Trained Tobit Models

Models were fitted on the Miami familial ALS cohort (`Convert` group, pre-onset visits only) and saved as slim RDS objects (data stripped). Two model sets are loaded:

- **Single-protein models** — one Tobit model per protein; formula: `Y ~ <Protein> + <Protein>_Init + X_Male + X_Geno_SOD1_A4V + X_Geno_SOD1_non_A4V + X_Geno_C9orf72`
- **Multi-protein panel models** — eight curated panels (Panel_1 through Panel_19, Panel_data, Panel_7_cox); same covariate structure with multiple proteins

The helper `inspect_model_proteins()` extracts the protein names expected by each model by reading the model's `terms` object, stripping `_Init` suffixes and covariate names.


In [ ]:
miami_model_dir <- here::here(
  "analysis", "05-Phenoconversion_timing_estimation", "Results", "1.Phenoconversion_timing_estimation"
)

single_models <- readRDS(file.path(miami_model_dir, "tobit_single_protein_models.rds"))
panel_models  <- readRDS(file.path(miami_model_dir, "tobit_multi_protein_models.rds"))

cat("Single-protein models: ", length(single_models), "\n")
cat("Multi-protein panels:  ", length(panel_models),  "\n")
cat("Panel names:", paste(names(panel_models), collapse = ", "), "\n")

# Extract protein names used by each model type
single_proteins <- names(single_models)

inspect_model_proteins <- function(model) {
  tt    <- tryCatch(terms(model), error = function(e) NULL)
  if (is.null(tt)) return(character(0))
  vars  <- attr(tt, "term.labels")
  prots <- unique(gsub("_Init$", "", vars))
  prots <- prots[!prots %in% c("X_Male", "X_Geno_SOD1_A4V",
                                "X_Geno_SOD1_non_A4V", "X_Geno_C9orf72")]
  prots
}

all_model_proteins <- unique(c(
  single_proteins,
  unlist(lapply(panel_models, inspect_model_proteins))
))

cat("\nTotal unique proteins across all models:", length(all_model_proteins), "\n")


## 4. Load UKB Data

Two files are loaded:

- **`delta_matrix_wide.csv`** — wide-format delta matrix (`eid` × proteins); missing values imputed to **0** before use
- **`visit_info.csv`** — UKB visit metadata including `Group`, `YrSinceDi`, `CollAge`, `Sex`, `GenoGroup`

Proteins required by the Miami models but absent from the UKB delta matrix are identified and will be set to **0** in the base input frame (consistent with the NA imputation applied to the full matrix above).

The 137 significantly differentially expressed proteins from the Miami DE analysis are also loaded to ensure any proteins absent from the delta matrix columns are explicitly zeroed out rather than left missing.


In [ ]:
delta_matrix_wide <- read.csv(here::here("data", "analysis_data", "ukb",
                                            "delta_matrix", "delta_matrix_wide.csv"))

# Fill the NA with 0
delta_matrix_wide[is.na(delta_matrix_wide)] <- 0

visit_info <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info",
                                   "visit_info.csv"), row.names = 1) %>%
  mutate(
    Group     = factor(Group, levels = c("Healthy control", "Pre-symptomatic",
                                         "Phenoconverter", "Pre-hospital", "Clinically manifest ALS")),
    GenoGroup = factor(GenoGroup),
    Sex       = factor(Sex, levels = c("Female", "Male"))
  )

available_in_ukb <- intersect(all_model_proteins, colnames(delta_matrix_wide))
missing_in_ukb   <- setdiff(all_model_proteins,   colnames(delta_matrix_wide))

cat("=== Group Counts ===\n")
print(table(visit_info$Group))
cat("\nProteins needed:        ", length(all_model_proteins), "\n")
cat("Available in UKB:       ", length(available_in_ukb),   "\n")
if (length(missing_in_ukb) > 0) {
  cat("Missing (set to NA):    ", length(missing_in_ukb),   "\n")
  cat(paste0("  ", missing_in_ukb, collapse = "\n"), "\n")
}


# DE results from Miami Analysis 1 — defines protein_sign
protein_de <- read.csv("../../01-Differentially_Expressed_Proteins/Results/Mix_effect_model_lmer.csv")
protein_137 <- protein_de %>%
  filter(significant == "Significant") %>%
  arrange(padj) %>%
  pull(Protein)

# for the protein not in the delta matrix, set the delta to 0
delta_matrix_wide[, setdiff(protein_137, colnames(delta_matrix_wide))] <- 0

## 5. Build Base UKB Input Frame

**Individuals included:** UKB **Convert** group with non-missing `YrSinceDi` only. This mirrors the Miami training population (Convert individuals with known time-to-onset).

**Covariate construction to match Miami model expectations:**

| Miami column | UKB construction |
|---|---|
| `<Protein>` | Delta value from `delta_matrix_wide` |
| `<Protein>_Init` | Same as current delta (UKB is cross-sectional — one visit only) |
| `X_Male` | `Sex == "Male"` → integer 0/1 |
| `X_Geno_SOD1_A4V` | `GenoGroup == "SOD1 A4V"` → 0/1 |
| `X_Geno_SOD1_non_A4V` | `GenoGroup == "SOD1 nonA4V"` → 0/1 |
| `X_Geno_C9orf72` | `GenoGroup == "C9orf72"` → 0/1 |

Proteins missing from UKB are added as `NA` columns (both current and `_Init`), which will return `NA` predictions for those models — handled gracefully by `compute_metrics`.


In [ ]:
# Affected + Convert (those with YrSinceDi)
vis_ukb <- visit_info %>%
  filter(Group %in% c("Phenoconverter", "Pre-hospital"), !is.na(YrSinceDi))

cat("Individuals with YrSinceDi:\n")
print(table(vis_ukb$Group))

# Base frame: merge delta matrix
df_base <- vis_ukb %>%
  select(eid, Group, YrSinceDi, CollAge, Sex, GenoGroup) %>%
  left_join(
    delta_matrix_wide %>% select(eid, any_of(available_in_ukb)),
    by = "eid"
  ) %>%
  mutate(
    X_Male              = as.integer(Sex == "Male"),
    X_Geno_SOD1_A4V     = ifelse(GenoGroup == "SOD1 A4V", 1, 0),
    X_Geno_SOD1_non_A4V = ifelse(GenoGroup == "SOD1 nonA4V", 1, 0),
    X_Geno_C9orf72       = ifelse(GenoGroup == "C9orf72", 1, 0)
  )

# Add _Init = current (cross-sectional)
for (p in available_in_ukb) {
  df_base[[paste0(p, "_Init")]] <- df_base[[p]]
}

# Add missing proteins as NA
for (p in missing_in_ukb) {
  df_base[[p]]                  <- NA_real_
  df_base[[paste0(p, "_Init")]] <- NA_real_
}

cat("\nBase frame: ", nrow(df_base), "rows x", ncol(df_base), "cols\n")


## 6. Apply Models Across All Horizons

For each horizon `h ∈ {−5, −4, −3, −2, −1, 0}`:

1. **Shift** each individual's time: `Y = YrSinceDi − h`
2. **Filter** to `Y < 0` — drops individuals whose shifted time reaches or exceeds diagnosis
3. **Predict** using all single-protein and panel models via `predict(..., type = "response")`
4. **Cap** predictions at 0: `pmin(pred_raw, 0)` — enforces the Tobit upper bound
5. **Compute metrics** (MAE, RMSE, Correlation, R²) comparing `y_pred` to the shifted `y_true`
6. **Store** predictions and metrics in long-format lists

**N varies by horizon** — stricter horizons (more negative `h`) retain fewer individuals because more individuals have `|YrSinceDi| < |h|` and are dropped by the filter.

**Outputs saved:**
- `Results/2.Phenoconversion_UKB_Horizons/metrics_all_horizons.csv`
- `data/analysis_data/ukb/predictions/predictions_all_horizons.csv`


In [ ]:
outdir <- file.path("Results", "2.Phenoconversion_UKB_Horizons")
ind_outdir <- here::here("data", "analysis_data", "ukb", "predictions")

dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

# Storage: all predictions (long format)
all_predictions <- list()   # one entry per horizon x model
all_metrics     <- list()   # metrics summary

options(repr.plot.width = 14, repr.plot.height = 20)

HORIZONS <- c(-5, -4, -3, -2, -1, 0)

for (h in HORIZONS) {
  # h is positive (1, 2, 3, 4, 5)
  # y_true = YrSinceDi + h  (shift each patient's time by h years)
  # Drop if y_true >= 0  (i.e. YrSinceDi >= -h)

  cat("\n", strrep("=", 60), "\n")
  cat("=== HORIZON:", h, "years before diagnosis ===\n")
  cat(strrep("=", 60), "\n")

  df_h <- df_base %>%
    mutate(Y = YrSinceDi - h) %>%   # shift each patient individually
    filter(Y < 0)                    # drop where shifted time >= 0

  cat("  Individuals passing filter:", nrow(df_h),
      "| y_true range:", round(min(df_h$Y), 2), "to", round(max(df_h$Y), 2), "\n")

  if (nrow(df_h) < 3) {
    cat("  Too few individuals — skipping horizon", h, "\n")
    next
  }

  # ── Single-protein models ────────────────────────────────────────────────
  for (protein in names(single_models)) {

    if (!protein %in% colnames(df_h)) next

    pred_raw <- tryCatch(
      suppressWarnings(predict(single_models[[protein]],
                               newdata = df_h, type = "response")),
      error = function(e) NULL
    )
    if (is.null(pred_raw)) next

    y_pred <- pmin(pred_raw, 0)
    y_true <- df_h$Y   # patient-specific shifted time

    m <- compute_metrics(y_true, y_pred)

    all_metrics[[length(all_metrics) + 1]] <- data.frame(
      Horizon    = h,
      Model      = protein,
      Model_type = "Single-Protein",
      N          = nrow(df_h),
      MAE        = m$MAE,
      RMSE       = m$RMSE,
      Correlation = m$cor_y,
      R2_Simple  = m$R_squared
    )

    all_predictions[[length(all_predictions) + 1]] <- data.frame(
      eid        = df_h$eid,
      Group      = df_h$Group,
      YrSinceDi  = df_h$YrSinceDi,
      Horizon    = h,
      Model      = protein,
      Model_type = "Single-Protein",
      y_true     = y_true,     # YrSinceDi + h, patient-specific
      y_pred     = y_pred,
      residual   = y_true - y_pred
    )
  }

  # ── Panel models ──────────────────────────────────────────────────────────
  for (panel_name in names(panel_models)) {

    pred_raw <- tryCatch(
      suppressWarnings(predict(panel_models[[panel_name]],
                               newdata = df_h, type = "response")),
      error = function(e) {
        cat("  Panel", panel_name, "failed at horizon", h, "\n"); NULL
      }
    )
    if (is.null(pred_raw)) next

    y_pred <- pmin(pred_raw, 0)
    y_true <- df_h$Y

    m <- compute_metrics(y_true, y_pred)

    all_metrics[[length(all_metrics) + 1]] <- data.frame(
      Horizon    = h,
      Model      = panel_name,
      Model_type = "Panel",
      N          = nrow(df_h),
      MAE        = m$MAE,
      RMSE       = m$RMSE,
      Correlation = m$cor_y,
      R2_Simple  = m$R_squared
    )

    all_predictions[[length(all_predictions) + 1]] <- data.frame(
      eid        = df_h$eid,
      Group      = df_h$Group,
      YrSinceDi  = df_h$YrSinceDi,
      Horizon    = h,
      Model      = panel_name,
      Model_type = "Panel",
      y_true     = y_true,
      y_pred     = y_pred,
      residual   = y_true - y_pred
    )
  }

  cat("  Horizon", h, "complete.\n")
}

# Combine
metrics_df     <- bind_rows(all_metrics)
predictions_df <- bind_rows(all_predictions)

write.csv(metrics_df,
          file.path(outdir, "metrics_all_horizons.csv"), row.names = FALSE)

write.csv(predictions_df,
          file.path(ind_outdir, "predictions_all_horizons.csv"), row.names = FALSE)

cat("\nAll horizons complete.\n")
cat("Total prediction rows: ", nrow(predictions_df), "\n")
cat("Metrics rows:          ", nrow(metrics_df),     "\n")


## 7. Diagnostic Plots Per Model Per Horizon

Six-panel diagnostic plots are generated for each combination of model × horizon, using the same layout as the Miami Rmd:

**Models plotted:**
- All multi-protein panel models
- Top 5 single-protein models ranked by mean MAE across all horizons

All plots are collected into a list and exported as a single PDF (`diagnostic_plots.pdf`) rather than printed inline, to avoid flooding the notebook output.


In [ ]:
# Plot all 6-panel diagnostics for each model x horizon combination
# (same layout as Miami Rmd)

options(repr.plot.width = 14, repr.plot.height = 20)

# Decide which models to plot: panels + top 5 single-proteins by overall MAE
top_single <- metrics_df %>%
  filter(Model_type == "Single-Protein") %>%
  group_by(Model) %>%
  summarise(mean_MAE = mean(MAE, na.rm = TRUE), .groups = "drop") %>%
  arrange(mean_MAE) %>%
  pull(Model)

models_to_plot <- c(names(panel_models), top_single)

cat("Models selected for diagnostic plots:\n")
cat(paste0("  ", models_to_plot, collapse = "\n"), "\n\n")

# create a list of plots
plots <- list()
for (model_name in models_to_plot) {
  for (h in HORIZONS) {

    sub <- predictions_df %>%
      filter(Model == model_name, Horizon == h, !is.na(y_pred))

    if (nrow(sub) < 3) next

    y_true <- sub$y_true
    y_pred <- sub$y_pred
    m      <- compute_metrics(y_true, y_pred)
    lbl    <- paste0(model_name, " | Horizon: ", h, " yr")

    p1 <- Bland_Altman_plot(y_true, y_pred, lbl)
    p2 <- residual_hist(y_true, y_pred, lbl)
    p3 <- predict_vs_observed_plot_tobit(
      y_true, y_pred, lbl, vjust = 1.0,
      MAE = m$MAE, RMSE = m$RMSE, cor_y = m$cor_y, R_squared = m$R_squared)
    p4 <- predict_vs_observed_boxplot(y_true, y_pred, lbl)
    p5 <- residual_vs_predict(y_true, y_pred, lbl)
    p6 <- residual_vs_predict_bin(y_true, y_pred, lbl)

    plots[[paste0(model_name, "_h", h)]] <- ((p1 | p2) / (p3 | p4) / (p5 | p6)) + plot_layout(heights = c(1, 1, 1))
  }
}


In [ ]:
# create a pdf to save the plots
pdf(file.path(outdir, "diagnostic_plots.pdf"), width = 14, height = 20)
for (plot in plots) {
  print(plot)
}
dev.off()


## 8. Performance Across Horizons

Three complementary visualisations of model performance across the six horizons:

1. **MAE heatmap** — models (rows) × horizons (columns), coloured blue (low MAE) to red (high MAE); models sorted by mean MAE across all horizons; faceted by model type (Panel / Single-Protein)

2. **MAE line plot** — trajectory of MAE as the horizon moves from −5 to 0 for all panel models and the top 5 single-protein models; solid lines = panels, dashed = single-proteins

3. **Residual distribution grid** — histograms of `residual = y_true − y_pred` for each panel model × horizon combination; a dashed vertical line at 0 marks perfect prediction


In [ ]:
# ── MAE heatmap: Model x Horizon ─────────────────────────────────────────────
options(repr.plot.width = 12, repr.plot.height = 8)

heat_df <- metrics_df %>%
  mutate(Horizon = factor(Horizon, levels = HORIZONS))

# Order models by mean MAE across all horizons
model_order <- heat_df %>%
  group_by(Model) %>%
  summarise(mean_MAE = mean(MAE, na.rm = TRUE), .groups = "drop") %>%
  arrange(mean_MAE) %>%
  pull(Model)

heat_df <- heat_df %>%
  mutate(Model = factor(Model, levels = rev(model_order)))

p_heat <- ggplot(heat_df, aes(x = Horizon, y = Model, fill = MAE)) +
  geom_tile(color = "white", linewidth = 0.4) +
  geom_text(aes(label = round(MAE, 2)), size = 3, color = "black") +
  facet_wrap(~Model_type, scales = "free_y", ncol = 2) +
  scale_fill_gradient2(low = "#4575B4", mid = "white", high = "#D73027",
                       midpoint = median(heat_df$MAE, na.rm = TRUE),
                       name = "MAE") +
  labs(title = "MAE by Model and Pre-Diagnosis Horizon (UKB)",
       subtitle = "Lower (blue) = better | Sorted by mean MAE across horizons",
       x = "Horizon (years before diagnosis)", y = NULL) +
  theme_minimal() +
  theme(plot.title    = element_text(face = "bold", hjust = 0.5),
        plot.subtitle = element_text(hjust = 0.5, color = "gray40"),
        axis.text.y   = element_text(size = 8),
        panel.grid    = element_blank(),
        strip.text    = element_text(face = "bold"))

print(p_heat)


In [ ]:
# ── Line plot: MAE trajectory across horizons ─────────────────────────────────
options(repr.plot.width = 11, repr.plot.height = 6)

top_models_line <- c(names(panel_models), top_single)

line_df <- metrics_df %>%
  filter(Model %in% top_models_line) %>%
  mutate(Horizon   = as.numeric(as.character(Horizon)),
         Model_type = factor(Model_type, levels = c("Panel", "Single-Protein")))

ggplot(line_df, aes(x = Horizon, y = MAE,
                    color = Model, linetype = Model_type, group = Model)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2.5) +
  scale_x_continuous(breaks = HORIZONS,
                     labels = paste0(HORIZONS, " yr")) +
  scale_linetype_manual(values = c("Panel" = "solid", "Single-Protein" = "dashed")) +
  labs(title    = "MAE Across Pre-Diagnosis Horizons",
       subtitle = "Each line = one model; solid = panel, dashed = single-protein",
       x = "Horizon (years before diagnosis)",
       y = "MAE (years)",
       color    = "Model",
       linetype = "Type") +
  theme_classic(base_size = 12) +
  theme(plot.title    = element_text(face = "bold", hjust = 0.5),
        plot.subtitle = element_text(hjust = 0.5, color = "gray40"),
        legend.position = "right")


In [ ]:
# ── Residual distribution ridge-style: one facet per horizon ─────────────────
options(repr.plot.width = 14, repr.plot.height = 15)

panel_pred <- predictions_df %>%
  filter(Model_type == "Panel") %>%
  mutate(Horizon_label = paste0(Horizon, " yr"),
         Horizon_label = factor(Horizon_label,
                                levels = paste0(HORIZONS, " yr")))

ggplot(panel_pred, aes(x = residual, fill = Model)) +
  geom_histogram(bins = 20, alpha = 0.7, position = "identity", color = "white") +
  geom_vline(xintercept = 0, linetype = "dashed", color = "black", linewidth = 0.8) +
  facet_grid(Model ~ Horizon_label, scales = "free_y") +
  labs(title    = "Residual Distributions by Panel and Horizon",
       subtitle = "Residual = Horizon - Predicted | Centred at 0 = perfect",
       x = "Residual (years)", y = "Count") +
  theme_bw(base_size = 11) +
  theme(plot.title    = element_text(face = "bold", hjust = 0.5),
        plot.subtitle = element_text(hjust = 0.5, color = "gray40"),
        strip.text    = element_text(size = 8),
        legend.position = "none")


## 9. Summary Tables

Three summary tables:

1. **Best model per horizon** — the single model with lowest MAE at each of the six horizons
2. **Panel model MAE wide table** — one row per panel, one column per horizon, plus mean MAE across horizons; sorted by mean MAE ascending
3. **Full metrics table** — all models × all horizons sorted by horizon then MAE; also saved to `metrics_summary_table.csv`


In [ ]:
# Best model per horizon
best_per_horizon <- metrics_df %>%
  group_by(Horizon) %>%
  slice_min(MAE, n = 1) %>%
  ungroup() %>%
  arrange(Horizon) %>%
  mutate(across(c(MAE, RMSE, Correlation, R2_Simple), ~ round(.x, 3)))

cat("=== Best Model Per Horizon ===\n")
knitr::kable(best_per_horizon,
             caption = "Best-Performing Model at Each Pre-Diagnosis Horizon (by MAE)")


In [ ]:
# Panel models summary across horizons (wide: one row per panel, cols = horizons)
panel_wide <- metrics_df %>%
  filter(Model_type == "Panel") %>%
  select(Model, Horizon, MAE) %>%
  pivot_wider(names_from = Horizon, values_from = MAE,
              names_prefix = "MAE_h") %>%
  mutate(Mean_MAE = rowMeans(across(starts_with("MAE_h")), na.rm = TRUE)) %>%
  arrange(Mean_MAE) %>%
  mutate(across(where(is.numeric), ~ round(.x, 3)))

cat("=== Panel Models — MAE Across Horizons ===\n")
knitr::kable(panel_wide,
             caption = "Panel Model MAE at Each Horizon (columns) + Mean MAE")


In [ ]:
# Full metrics table sorted by MAE
full_table <- metrics_df %>%
  arrange(Horizon, MAE) %>%
  mutate(across(c(MAE, RMSE, Correlation, R2_Simple), ~ round(.x, 3)))

write.csv(full_table,
          file.path(outdir, "metrics_summary_table.csv"), row.names = FALSE)

knitr::kable(full_table,
             caption = "All Models × All Horizons — Performance Metrics")


## 10. Output Files

| File | Location | Description |
|------|----------|-------------|
| `metrics_all_horizons.csv` | `Results/2.Phenoconversion_UKB_Horizons/` | MAE, RMSE, Correlation, R² for every model × horizon |
| `metrics_summary_table.csv` | `Results/2.Phenoconversion_UKB_Horizons/` | Same, formatted and sorted |
| `diagnostic_plots.pdf` | `Results/2.Phenoconversion_UKB_Horizons/` | 6-panel diagnostic plots for selected models × horizons |
| `predictions_all_horizons.csv` | `data/analysis_data/ukb/predictions/` | Per-individual predictions for every model × horizon |

### Column Definitions

| Column | Meaning |
|--------|---------|
| `Horizon` | The shift value `h` applied (one of −5, −4, −3, −2, −1, 0) |
| `YrSinceDi` | Individual's original years since diagnosis (negative = pre-diagnosis) |
| `y_true` | Shifted ground truth: `YrSinceDi − h` (patient-specific, always < 0) |
| `y_pred` | Model predicted `YrSinceOs`, capped at 0 via `pmin(..., 0)` |
| `residual` | `y_true − y_pred` (positive = model predicts earlier than shifted truth) |
| `MAE` | Mean absolute error between `y_true` and `y_pred` across individuals |
| `Correlation` | Pearson correlation between `y_pred` and `y_true` across individuals |
| `R2_Simple` | 1 − SS_res / SS_tot (can be negative if model worse than mean prediction) |
